In [0]:
dbutils.widgets.text("catalog", "tpch_sales")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

region_df = spark.read.table(f"{catalog}.bronze.region")
cleaned_region = region_df.withColumnRenamed("r_regionkey", "regionkey") \
    .withColumnRenamed("r_name", "name") \
    .withColumnRenamed("r_comment", "comment")

bronze_count = cleaned_region.count()

w = Window.partitionBy("regionkey").orderBy(lit(1)) #PK validation
cleaned_region = cleaned_region.withColumn("rn", row_number().over(w))

# Quarantine
quarantine_region = cleaned_region.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("regionkey").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn")

# Valid
cleaned_region = cleaned_region.select("regionkey", "name","comment").where("regionkey is not null and rn = 1").drop("rn")

cleaned_region.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region")
quarantine_region.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_region")

# Reconciliation
print(f"Region - Bronze: {bronze_count}, Valid: {cleaned_region.count()}, Quarantine: {quarantine_region.count()}, Sum: {cleaned_region.count() + quarantine_region.count()}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

nation_df = spark.read.table(f"{catalog}.bronze.nation")
cleaned_nation = nation_df.withColumnRenamed("n_nationkey", "nationkey") \
    .withColumnRenamed("n_name", "name") \
    .withColumnRenamed("n_regionkey", "regionkey") \
    .withColumnRenamed("n_comment", "comment")

bronze_count = cleaned_nation.count()

w = Window.partitionBy("nationkey").orderBy(lit(1))
cleaned_nation = cleaned_nation.withColumn("rn", row_number().over(w))

# Referential Integrity
valid_region = spark.read.table(f"{catalog}.silver.region").select("regionkey").withColumnRenamed("regionkey", "_valid_regionkey")
cleaned_nation = cleaned_nation.join(valid_region, cleaned_nation.regionkey == valid_region._valid_regionkey, "left")

# Quarantine
quarantine_nation = cleaned_nation.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("nationkey").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_regionkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn", "_valid_regionkey")

# Valid
cleaned_nation = cleaned_nation.select("nationkey", "name", "regionkey", "comment").where("nationkey is not null and rn = 1 and _valid_regionkey is not null") \
    .drop("rn", "_valid_regionkey")

cleaned_nation.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation")
quarantine_nation.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_nation")

# Reconciliation
print(f"Nation - Bronze: {bronze_count}, Valid: {cleaned_nation.count()}, Quarantine: {quarantine_nation.count()}, Sum: {cleaned_nation.count() + quarantine_nation.count()}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

customers_df = spark.read.table(f"{catalog}.bronze.customer")
cleaned_customer = customers_df.withColumnRenamed("c_custkey", "custkey").withColumnRenamed("c_name", "name").withColumnRenamed("c_address", "address").withColumnRenamed("c_nationkey", "nationkey").withColumnRenamed("c_phone", "phone").withColumnRenamed("c_acctbal", "acctbal").withColumnRenamed("c_mktsegment", "mktsegment").withColumnRenamed("c_comment", "comment")

bronze_count = cleaned_customer.count()

w = Window.partitionBy("custkey").orderBy(lit(1))
cleaned_customer = cleaned_customer.withColumn("rn", row_number().over(w))

# Referential Integrity
valid_nation = spark.read.table(f"{catalog}.silver.nation").select("nationkey").withColumnRenamed("nationkey", "_valid_nationkey")
cleaned_customer = cleaned_customer.join(valid_nation, cleaned_customer.nationkey == valid_nation._valid_nationkey, "left")

# Quarantine
quarantine_customer = cleaned_customer.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("custkey").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_nationkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn", "_valid_nationkey")

# Valid
cleaned_customer = cleaned_customer.select("custkey", "name", "address", "nationkey", "phone", "acctbal", "mktsegment", "comment").where("custkey is not null and rn = 1 and _valid_nationkey is not null") \
    .drop("rn", "_valid_nationkey")

cleaned_customer.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customer")
quarantine_customer.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_customer")

# Reconciliation
print(f"Customer - Bronze: {bronze_count}, Valid: {cleaned_customer.count()}, Quarantine: {quarantine_customer.count()}, Sum: {cleaned_customer.count() + quarantine_customer.count()}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

suppliers_df = spark.read.table(f"{catalog}.bronze.supplier")
cleaned_supplier = suppliers_df.withColumnRenamed("s_suppkey", "suppkey").withColumnRenamed("s_name", "name").withColumnRenamed("s_address", "address").withColumnRenamed("s_nationkey", "nationkey").withColumnRenamed("s_phone","phone").withColumnRenamed("s_acctbal", "acctbal").withColumnRenamed("s_comment", "comment")

bronze_count = cleaned_supplier.count()

w = Window.partitionBy("suppkey").orderBy(lit(1))
cleaned_supplier = cleaned_supplier.withColumn("rn", row_number().over(w))

# Referential Integrity
valid_nation = spark.read.table(f"{catalog}.silver.nation").select("nationkey").withColumnRenamed("nationkey", "_valid_nationkey")
cleaned_supplier = cleaned_supplier.join(valid_nation, cleaned_supplier.nationkey == valid_nation._valid_nationkey, "left")

# Quarantine
quarantine_supplier = cleaned_supplier.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("suppkey").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_nationkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn", "_valid_nationkey")

# Valid
cleaned_supplier = cleaned_supplier.select("suppkey", "name", "address", "nationkey", "phone", "acctbal", "comment").where("suppkey is not null and rn = 1 and _valid_nationkey is not null") \
    .drop("rn", "_valid_nationkey")

cleaned_supplier.write.mode("overwrite").saveAsTable(f"{catalog}.silver.supplier")
quarantine_supplier.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_supplier")

# Reconciliation: valid + quarantine must equal bronze
print(f"Supplier - Bronze: {bronze_count}, Valid: {cleaned_supplier.count()}, Quarantine: {quarantine_supplier.count()}, Sum: {cleaned_supplier.count() + quarantine_supplier.count()}")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

parts_df = spark.read.table(f"{catalog}.bronze.part")
cleaned_part = parts_df.withColumnRenamed("p_partkey", "partkey").withColumnRenamed("p_name", "name").withColumnRenamed("p_mfgr", "mfgr").withColumnRenamed("p_brand", "brand").withColumnRenamed("p_type", "type").withColumnRenamed("p_size", "size").withColumnRenamed("p_container", "container").withColumnRenamed("p_retailprice", "retailprice").withColumnRenamed("p_comment", "comment")

bronze_count = cleaned_part.count()

w = Window.partitionBy("partkey").orderBy(lit(1))
cleaned_part = cleaned_part.withColumn("rn", row_number().over(w))

# Quarantine
quarantine_part = cleaned_part.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("partkey").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn")

# Valid
cleaned_part = cleaned_part.select("partkey", "name", "mfgr", "brand", "type", "size", "container", "retailprice", "comment").where("partkey is not null and rn = 1") \
    .drop("rn")

cleaned_part.write.mode("overwrite").saveAsTable(f"{catalog}.silver.part")
quarantine_part.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_part")

# Reconciliation
print(f"Part - Bronze: {bronze_count}, Valid: {cleaned_part.count()}, Quarantine: {quarantine_part.count()}, Sum: {cleaned_part.count() + quarantine_part.count()}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

orders_df = spark.read.table(f"{catalog}.bronze.orders")
cleaned_orders = orders_df.withColumnRenamed("o_orderkey", "orderkey") \
    .withColumnRenamed("o_custkey", "custkey") \
    .withColumnRenamed("o_orderstatus", "orderstatus") \
    .withColumnRenamed("o_totalprice", "totalprice") \
    .withColumnRenamed("o_orderdate", "orderdate") \
    .withColumnRenamed("o_orderpriority", "orderpriority") \
    .withColumnRenamed("o_clerk", "clerk") \
    .withColumnRenamed("o_shippriority", "shippriority") \
    .withColumnRenamed("o_comment", "comment")

bronze_count = cleaned_orders.count()

w = Window.partitionBy("orderkey").orderBy(lit(1))
cleaned_orders = cleaned_orders.withColumn("rn", row_number().over(w))

#Referential Integrity
valid_customer = spark.read.table(f"{catalog}.silver.customer").select("custkey").withColumnRenamed("custkey", "_valid_custkey")
cleaned_orders = cleaned_orders.join(valid_customer, cleaned_orders.custkey == valid_customer._valid_custkey, "left")

# orderdate must be between 1992-01-01 and 1998-12-31

# Quarantine
quarantine_orders = cleaned_orders.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("orderkey").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_custkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("orderdate").isNull() | (col("orderdate") < to_date(lit("1992-01-01"))) | (col("orderdate") > to_date(lit("1998-12-31"))), concat(col("rejection_reason"), lit("DATE_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn", "_valid_custkey")

# Valid
cleaned_orders = cleaned_orders.select("orderkey", "custkey", "orderstatus", "totalprice", "orderdate", "orderpriority", "clerk", "shippriority", "comment") \
    .where("orderkey is not null and rn = 1 and _valid_custkey is not null and orderdate >= '1992-01-01' and orderdate <= '1998-12-31'") \
    .drop("rn", "_valid_custkey")

cleaned_orders.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders")
quarantine_orders.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_orders")

# Reconciliation
print(f"Orders - Bronze: {bronze_count}, Valid: {cleaned_orders.count()}, Quarantine: {quarantine_orders.count()}, Sum: {cleaned_orders.count() + quarantine_orders.count()}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

partsupp_df = spark.read.table(f"{catalog}.bronze.partsupp")
cleaned_partsupp = partsupp_df.withColumnRenamed("ps_partkey", "partkey") \
    .withColumnRenamed("ps_suppkey", "suppkey") \
    .withColumnRenamed("ps_availqty", "availqty") \
    .withColumnRenamed("ps_supplycost", "supplycost") \
    .withColumnRenamed("ps_comment", "comment")

bronze_count = cleaned_partsupp.count()

w = Window.partitionBy("partkey", "suppkey").orderBy(lit(1))
cleaned_partsupp = cleaned_partsupp.withColumn("rn", row_number().over(w))

# Referential Integrity: partkey must exist in silver.part 
valid_part = spark.read.table(f"{catalog}.silver.part").select("partkey").withColumnRenamed("partkey", "_valid_partkey")
cleaned_partsupp = cleaned_partsupp.join(valid_part, cleaned_partsupp.partkey == valid_part._valid_partkey, "left")

# Referential Integrity: suppkey must exist in silver.supplier
valid_supplier = spark.read.table(f"{catalog}.silver.supplier").select("suppkey").withColumnRenamed("suppkey", "_valid_suppkey")
cleaned_partsupp = cleaned_partsupp.join(valid_supplier, cleaned_partsupp.suppkey == valid_supplier._valid_suppkey, "left")

# Quarantine
quarantine_partsupp = cleaned_partsupp.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("partkey").isNull() | col("suppkey").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_partkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_suppkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn", "_valid_partkey", "_valid_suppkey")

# Valid
cleaned_partsupp = cleaned_partsupp.select("partkey", "suppkey", "availqty", "supplycost", "comment") \
    .where("partkey is not null and suppkey is not null and rn = 1 and _valid_partkey is not null and _valid_suppkey is not null") \
    .drop("rn", "_valid_partkey", "_valid_suppkey")

cleaned_partsupp.write.mode("overwrite").saveAsTable(f"{catalog}.silver.partsupp")
quarantine_partsupp.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_partsupp")

# Reconciliation
print(f"PartSupp - Bronze: {bronze_count}, Valid: {cleaned_partsupp.count()}, Quarantine: {quarantine_partsupp.count()}, Sum: {cleaned_partsupp.count() + quarantine_partsupp.count()}")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

lineitem_df = spark.read.table(f"{catalog}.bronze.lineitem")
cleaned_lineitem = lineitem_df.withColumnRenamed("l_orderkey", "orderkey") \
    .withColumnRenamed("l_partkey", "partkey") \
    .withColumnRenamed("l_suppkey", "suppkey") \
    .withColumnRenamed("l_linenumber", "linenumber") \
    .withColumnRenamed("l_quantity", "quantity") \
    .withColumnRenamed("l_extendedprice", "extendedprice") \
    .withColumnRenamed("l_discount", "discount") \
    .withColumnRenamed("l_tax", "tax") \
    .withColumnRenamed("l_returnflag","returnflag") \
    .withColumnRenamed("l_linestatus","linestatus") \
    .withColumnRenamed("l_shipdate", "shipdate") \
    .withColumnRenamed("l_commitdate", "commitdate") \
    .withColumnRenamed("l_receiptdate", "receiptdate") \
    .withColumnRenamed("l_shipinstruct", "shipinstruct") \
    .withColumnRenamed("l_shipmode", "shipmode") \
    .withColumnRenamed("l_comment", "comment")

bronze_count = cleaned_lineitem.count()

# Primary Key Validation: composite key (orderkey, linenumber) 
w = Window.partitionBy("orderkey", "linenumber").orderBy(lit(1))
cleaned_lineitem = cleaned_lineitem.withColumn("rn", row_number().over(w))

# Referential Integrity: orderkey must exist in silver.orders
valid_orders = spark.read.table(f"{catalog}.silver.orders").select("orderkey").withColumnRenamed("orderkey", "_valid_orderkey")
cleaned_lineitem = cleaned_lineitem.join(valid_orders, cleaned_lineitem.orderkey == valid_orders._valid_orderkey, "left")

# Referential Integrity: partkey must exist in silver.part
valid_part = spark.read.table(f"{catalog}.silver.part").select("partkey").withColumnRenamed("partkey", "_valid_partkey")
cleaned_lineitem = cleaned_lineitem.join(valid_part, cleaned_lineitem.partkey == valid_part._valid_partkey, "left")

# Referential Integrity: suppkey must exist in silver.supplier
valid_supplier = spark.read.table(f"{catalog}.silver.supplier").select("suppkey").withColumnRenamed("suppkey", "_valid_suppkey")
cleaned_lineitem = cleaned_lineitem.join(valid_supplier, cleaned_lineitem.suppkey == valid_supplier._valid_suppkey, "left")


# Quarantine
quarantine_lineitem = cleaned_lineitem.withColumn("rejection_reason", lit("")) \
    .withColumn("rejection_reason", when(col("orderkey").isNull() | col("linenumber").isNull(), concat(col("rejection_reason"), lit("NULL_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("rn") > 1, concat(col("rejection_reason"), lit("DUPLICATE_PRIMARY_KEY;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_orderkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_partkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("_valid_suppkey").isNull(), concat(col("rejection_reason"), lit("REFERENTIAL_INTEGRITY_FAIL;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("quantity").isNull() | (col("quantity") <= 0), concat(col("rejection_reason"), lit("NUMERIC_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("extendedprice").isNull() | (col("extendedprice") <= 0), concat(col("rejection_reason"), lit("NUMERIC_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("discount").isNull() | (col("discount") < 0) | (col("discount") > 0.10), concat(col("rejection_reason"), lit("NUMERIC_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("tax").isNull() | (col("tax") < 0) | (col("tax") > 0.08), concat(col("rejection_reason"), lit("NUMERIC_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("shipdate").isNull() | (col("shipdate") < to_date(lit("1992-01-01"))) | (col("shipdate") > to_date(lit("1998-12-31"))), concat(col("rejection_reason"), lit("DATE_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("commitdate").isNull() | (col("commitdate") < to_date(lit("1992-01-01"))) | (col("commitdate") > to_date(lit("1998-12-31"))), concat(col("rejection_reason"), lit("DATE_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .withColumn("rejection_reason", when(col("receiptdate").isNull() | (col("receiptdate") < to_date(lit("1992-01-01"))) | (col("receiptdate") > to_date(lit("1998-12-31"))), concat(col("rejection_reason"), lit("DATE_RANGE_VIOLATION;"))).otherwise(col("rejection_reason"))) \
    .where("rejection_reason != ''") \
    .drop("rn", "_valid_orderkey", "_valid_partkey", "_valid_suppkey")

# Valid
cleaned_lineitem = cleaned_lineitem.select("orderkey", "partkey", "suppkey", "linenumber", "quantity", "extendedprice", "discount", "tax", "returnflag", "linestatus", "shipdate", "commitdate", "receiptdate", "shipinstruct", "shipmode", "comment") \
    .where("orderkey is not null and linenumber is not null and rn = 1 and _valid_orderkey is not null and _valid_partkey is not null and _valid_suppkey is not null and quantity > 0 and extendedprice > 0 and discount >= 0 and discount <= 0.10 and tax >= 0 and tax <= 0.08 and shipdate >= '1992-01-01' and shipdate <= '1998-12-31' and commitdate >= '1992-01-01' and commitdate <= '1998-12-31' and receiptdate >= '1992-01-01' and receiptdate <= '1998-12-31'") \
    .drop("rn", "_valid_orderkey", "_valid_partkey", "_valid_suppkey")

cleaned_lineitem.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem")
quarantine_lineitem.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_lineitem")

# Reconciliation
print(f"LineItem - Bronze: {bronze_count}, Valid: {cleaned_lineitem.count()}, Quarantine: {quarantine_lineitem.count()}, Sum: {cleaned_lineitem.count() + quarantine_lineitem.count()}")


In [0]:
from pyspark.sql.functions import col

tables = ["region", "nation", "customer", "supplier", "part", "orders", "partsupp", "lineitem"]

# Reconciliation: valid + quarantine must equal bronze
recon = []
for t in tables:
    b = spark.table(f"{catalog}.bronze.{t}").count()
    s = spark.table(f"{catalog}.silver.{t}").count()
    q = spark.table(f"{catalog}.silver.quarantine_{t}").count()
    recon.append((t, b, s, q, s + q, "PASS" if s + q == b else "FAIL"))
print("RECONCILIATION SUMMARY")
display(spark.createDataFrame(recon, ["Table", "Bronze", "Silver", "Quarantine", "Sum", "Status"]))

# Data Quality Report: failing records by validation category per table
reasons = ["NULL_PRIMARY_KEY", "DUPLICATE_PRIMARY_KEY", "REFERENTIAL_INTEGRITY_FAIL", "NUMERIC_RANGE_VIOLATION", "DATE_RANGE_VIOLATION"]
sql_parts = []
for t in tables:
    cols = ", ".join([f"SUM(CASE WHEN rejection_reason LIKE '%{r};%' THEN 1 ELSE 0 END) AS `{r}`" for r in reasons])
    sql_parts.append(f"SELECT '{t}' AS Table, {cols}, COUNT(*) AS total_quarantined FROM {catalog}.silver.quarantine_{t}")
print("\n DATA QUALITY REPORT ")
display(spark.sql(" UNION ALL ".join(sql_parts)))